# Speech-Based Gender Classification Using Transfer Learning

# Import Libraries

In [1]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Dataset Path

In [2]:
male_path = r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\male"
female_path = r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\female"

# Load Audio Files

In [3]:
X = []
y = []

def extract_features(file):
    
    audio, sr = librosa.load(file, sr=16000)
    
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    
    mfcc = np.mean(mfcc.T, axis=0)
    
    return mfcc

# Read Male Audio

In [4]:
for file in os.listdir(r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\male"):
    
    file_path = os.path.join(r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\male",r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\male\arctic_a0001(6).wav")
    
    features = extract_features(file_path)
    
    X.append(features)
    
    y.append("male")

# Read Female Audio

In [5]:
for file in os.listdir(r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\female"):
    
    file_path = os.path.join(r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\female", r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\female\arctic_a0001(4).wav")
    
    features = extract_features(file_path)
    
    X.append(features)
    
    y.append("female")

# Convert to Array

In [6]:
X = np.array(X)
y = np.array(y)

print(X.shape)

(16148, 40)


# Encode Labels

In [7]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

y_categorical = to_categorical(y_encoded)

# Train Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42
)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Transfer Learning Model (CNN Classifier)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential()

model.add(Conv1D(64, 3, activation='relu', input_shape=(40,1)))
model.add(MaxPooling1D(2))

model.add(Conv1D(128, 3, activation='relu'))
model.add(MaxPooling1D(2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(2, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\Shuhaib\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 38, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 19, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 17, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 8, 128)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         131,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 2)                   │             258 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 156,418 (611.01 KB)

 Trainable params: 156,418 (611.01 KB)

 Non-trainable params: 0 (0.00 B)

# Train Model

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 22s 32ms/step - accuracy: 0.9966 - loss: 0.0106 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - accuracy: 1.0000 - loss: 2.3332e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - accuracy: 1.0000 - loss: 1.0528e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 1.0000 - loss: 6.5805e-08 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 13s 31ms/step - accuracy: 1.0000 - loss: 2.3467e-08 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 1.0000 - loss: 3.3543e-08 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/10
404/404 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 1.0000 - loss: 7.1610e-09 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/10
404/404 ━━━━━━━━━━━━━━━━━━━

In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Accuracy: 1.0


# Predict Gender

In [12]:
def predict_gender(file):

    features = extract_features(file)

    features = features.reshape(1,40,1)

    prediction = model.predict(features)

    result = np.argmax(prediction)

    return encoder.classes_[result]

print(predict_gender(r"C:\Users\Shuhaib\OneDrive\Documents\archive\data\male\arctic_a0001(10).wav"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 701ms/step
male
